## Data Comparison

Compares datasets between two cycles, to identify changes and trends.

In [ ]:
try:
    import requests
    import sqlite3
    import pandas
    print("✓ All required dependencies are available")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("Run: poetry install")

In [ ]:
# Imports and setup
import os
import sqlite3
from pathlib import Path
from typing import Dict, List, Any
import requests
from datetime import datetime

# Configuration
DATA_DIR = Path('/workspace/data')
DATA_DIR.mkdir(exist_ok=True)

RELEASES = {
    'e6c2': {
        'tag': 'e6c2',
        'name': 'Era 6, Cycle 2',
        'url': 'https://github.com/Scetrov/evefrontier_datasets/releases/download/e6c2/static_data.db',
        'path': DATA_DIR / 'static_data_e6c2.db'
    },
    'e6c3': {
        'tag': 'e6c3',
        'name': 'Era 6, Cycle 3',
        'url': 'https://github.com/Scetrov/evefrontier_datasets/releases/download/e6c3/static_data.db',
        'path': DATA_DIR / 'static_data_e6c3.db'
    }
}

print(f"📁 Data directory: {DATA_DIR}")
print(f"📦 Releases to download: {', '.join([v['name'] for v in RELEASES.values()])}")

## Download Datasets

Download both static_data.db files from the releases.

In [ ]:
def download_file(url: str, destination: Path, force: bool = False) -> bool:
    """Download a file from URL to destination."""
    if destination.exists() and not force:
        print(f"✓ File already exists: {destination.name}")
        return True
    
    try:
        print(f"⬇️  Downloading {destination.name}...")
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        downloaded = 0
        
        with open(destination, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size:
                        percent = (downloaded / total_size) * 100
                        print(f"  Progress: {percent:.1f}% ({downloaded / 1024 / 1024:.1f} MB)", end='\r')
        
        print(f"\n✓ Downloaded: {destination.name} ({destination.stat().st_size / 1024 / 1024:.2f} MB)")
        return True
    except Exception as e:
        print(f"✗ Failed to download {destination.name}: {e}")
        return False

# Download both datasets
for release_key, release_info in RELEASES.items():
    download_file(release_info['url'], release_info['path'])

## Database Structure Analysis

Examine the structure and content of both databases.

In [ ]:
def get_database_info(db_path: Path) -> Dict[str, Any]:
    """Get comprehensive information about a database."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]
    
    info = {
        'path': str(db_path),
        'size_mb': db_path.stat().st_size / 1024 / 1024,
        'tables': {},
        'total_rows': 0
    }
    
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        row_count = cursor.fetchone()[0]
        
        cursor.execute(f"PRAGMA table_info({table})")
        columns = cursor.fetchall()
        
        info['tables'][table] = {
            'row_count': row_count,
            'columns': {col[1]: col[2] for col in columns}  # name: type
        }
        info['total_rows'] += row_count
    
    conn.close()
    return info

# Get info for both databases
db_info = {}
for release_key, release_info in RELEASES.items():
    db_path = release_info['path']
    if db_path.exists():
        db_info[release_key] = get_database_info(db_path)
        print(f"\n📊 {release_info['name']} Database")
        print(f"   File size: {db_info[release_key]['size_mb']:.2f} MB")
        print(f"   Total tables: {len(db_info[release_key]['tables'])}")
        print(f"   Total rows: {db_info[release_key]['total_rows']:,}")
        print(f"   Tables: {', '.join(db_info[release_key]['tables'].keys())}")
    else:
        print(f"✗ Database not found: {db_path}")

## Exhaustive Data Comparison

Perform detailed comparison between the two datasets.

In [ ]:
class DatasetComparator:
    """Compare two SQLite databases exhaustively."""
    
    def __init__(self, db1_path: Path, db2_path: Path):
        self.db1_path = db1_path
        self.db2_path = db2_path
        self.conn1 = sqlite3.connect(db1_path)
        self.conn2 = sqlite3.connect(db2_path)
        self.results = {
            'tables': {},
            'summary': {}
        }
    
    def compare_schemas(self) -> Dict[str, Any]:
        """Compare table schemas between databases."""
        cursor1 = self.conn1.cursor()
        cursor2 = self.conn2.cursor()
        
        cursor1.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables1 = set(row[0] for row in cursor1.fetchall())
        
        cursor2.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables2 = set(row[0] for row in cursor2.fetchall())
        
        schema_diff = {
            'only_in_db1': tables1 - tables2,
            'only_in_db2': tables2 - tables1,
            'in_both': tables1 & tables2,
            'column_differences': {}
        }
        
        # Compare columns for tables in both
        for table in schema_diff['in_both']:
            cursor1.execute(f"PRAGMA table_info({table})")
            cols1 = {row[1]: row[2] for row in cursor1.fetchall()}
            
            cursor2.execute(f"PRAGMA table_info({table})")
            cols2 = {row[1]: row[2] for row in cursor2.fetchall()}
            
            if cols1 != cols2:
                schema_diff['column_differences'][table] = {
                    'only_in_db1': set(cols1.keys()) - set(cols2.keys()),
                    'only_in_db2': set(cols2.keys()) - set(cols1.keys()),
                    'type_changes': {k: (cols1[k], cols2[k]) for k in cols1 if k in cols2 and cols1[k] != cols2[k]}
                }
        
        return schema_diff
    
    def compare_row_counts(self) -> Dict[str, Dict[str, int]]:
        """Compare row counts for each table."""
        cursor1 = self.conn1.cursor()
        cursor2 = self.conn2.cursor()
        
        row_counts = {}
        
        cursor1.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = [row[0] for row in cursor1.fetchall()]
        
        for table in tables:
            cursor1.execute(f"SELECT COUNT(*) FROM {table}")
            count1 = cursor1.fetchone()[0]
            
            try:
                cursor2.execute(f"SELECT COUNT(*) FROM {table}")
                count2 = cursor2.fetchone()[0]
            except sqlite3.OperationalError:
                count2 = None
            
            row_counts[table] = {
                'db1': count1,
                'db2': count2,
                'difference': count2 - count1 if count2 is not None else None,
                'percent_change': ((count2 - count1) / count1 * 100) if count1 > 0 and count2 is not None else None
            }
        
        return row_counts
    
    def compare_data_samples(self, limit: int = 5) -> Dict[str, Dict[str, Any]]:
        """Sample and compare actual data from tables."""
        cursor1 = self.conn1.cursor()
        cursor2 = self.conn2.cursor()
        
        samples = {}
        
        cursor1.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = [row[0] for row in cursor1.fetchall()]
        
        for table in tables:
            try:
                cursor1.execute(f"SELECT * FROM {table} LIMIT {limit}")
                rows1 = cursor1.fetchall()
                
                cursor2.execute(f"SELECT * FROM {table} LIMIT {limit}")
                rows2 = cursor2.fetchall()
                
                samples[table] = {
                    'db1_sample': rows1,
                    'db2_sample': rows2,
                    'samples_equal': rows1 == rows2
                }
            except Exception as e:
                samples[table] = {'error': str(e)}
        
        return samples
    
    def run_full_comparison(self) -> Dict[str, Any]:
        """Run all comparisons and generate report."""
        print("🔍 Starting exhaustive data comparison...\n")
        
        print("1️⃣  Comparing schemas...")
        schema_diff = self.compare_schemas()
        
        print("2️⃣  Comparing row counts...")
        row_counts = self.compare_row_counts()
        
        print("3️⃣  Sampling and comparing data...")
        samples = self.compare_data_samples()
        
        self.results = {
            'schema_differences': schema_diff,
            'row_count_comparison': row_counts,
            'data_samples': samples
        }
        
        return self.results
    
    def print_summary(self):
        """Print a human-readable summary of comparison results."""
        if not self.results:
            print("No comparison results available")
            return
        
        schema = self.results['schema_differences']
        print("\n" + "="*60)
        print("📋 SCHEMA COMPARISON SUMMARY")
        print("="*60)
        
        if schema['only_in_db1']:
            print(f"\n📌 Tables only in DB1: {schema['only_in_db1']}")
        if schema['only_in_db2']:
            print(f"📌 Tables only in DB2: {schema['only_in_db2']}")
        
        print(f"\n✓ Common tables: {len(schema['in_both'])}")
        
        if schema['column_differences']:
            print(f"\n⚠️  Tables with column differences:")
            for table, diffs in schema['column_differences'].items():
                print(f"   - {table}:")
                if diffs['only_in_db1']:
                    print(f"     Only in DB1: {diffs['only_in_db1']}")
                if diffs['only_in_db2']:
                    print(f"     Only in DB2: {diffs['only_in_db2']}")
                if diffs['type_changes']:
                    print(f"     Type changes: {diffs['type_changes']}")
        
        print("\n" + "="*60)
        print("📊 ROW COUNT COMPARISON")
        print("="*60)
        
        row_counts = self.results['row_count_comparison']
        changed_tables = {t: c for t, c in row_counts.items() if c['difference'] != 0}
        
        if changed_tables:
            print(f"\n🔄 Tables with row count changes ({len(changed_tables)}):")
            for table, counts in changed_tables.items():
                diff = counts['difference']
                pct = counts['percent_change']
                direction = "↑" if diff > 0 else "↓"
                print(f"   {direction} {table:30} | DB1: {counts['db1']:>8} → DB2: {counts['db2']:>8} ({pct:+.1f}%)")
        else:
            print("\n✓ All row counts identical")
        
        print("\n" + "="*60)

# Run the comparison
if 'e6c2' in db_info and 'e6c3' in db_info:
    comparator = DatasetComparator(RELEASES['e6c2']['path'], RELEASES['e6c3']['path'])
    comparison_results = comparator.run_full_comparison()
    comparator.print_summary()
else:
    print("✗ Cannot perform comparison: Not all databases available")

## Detailed Analysis by Table

Deep dive into specific table comparisons.

In [ ]:
def get_table_diff_analysis(table: str, db1_path: Path, db2_path: Path) -> Dict[str, Any]:
    """Analyze detailed differences for a specific table."""
    conn1 = sqlite3.connect(db1_path)
    conn2 = sqlite3.connect(db2_path)
    
    cursor1 = conn1.cursor()
    cursor2 = conn2.cursor()
    
    # Get primary key info
    cursor1.execute(f"PRAGMA table_info({table})")
    columns = [row[1] for row in cursor1.fetchall()]
    
    cursor1.execute(f"PRAGMA table_info({table})")
    pk_info = cursor1.fetchall()
    pk_column = next((row[1] for row in pk_info if row[5]), None)
    
    analysis = {
        'table': table,
        'columns': columns,
        'primary_key': pk_column,
        'db1_all_ids': set(),
        'db2_all_ids': set(),
        'added_rows': [],
        'removed_rows': [],
        'modified_rows': []
    }
    
    if pk_column:
        # Get all IDs from both databases
        cursor1.execute(f"SELECT DISTINCT {pk_column} FROM {table}")
        analysis['db1_all_ids'] = {row[0] for row in cursor1.fetchall()}
        
        cursor2.execute(f"SELECT DISTINCT {pk_column} FROM {table}")
        analysis['db2_all_ids'] = {row[0] for row in cursor2.fetchall()}
        
        # Find added and removed rows
        analysis['removed_rows'] = list(analysis['db1_all_ids'] - analysis['db2_all_ids'])
        analysis['added_rows'] = list(analysis['db2_all_ids'] - analysis['db1_all_ids'])
        
        # Check for modifications in common rows
        common_ids = analysis['db1_all_ids'] & analysis['db2_all_ids']
        for row_id in list(common_ids)[:10]:  # Sample first 10
            cursor1.execute(f"SELECT * FROM {table} WHERE {pk_column} = ?", (row_id,))
            row1 = cursor1.fetchone()
            
            cursor2.execute(f"SELECT * FROM {table} WHERE {pk_column} = ?", (row_id,))
            row2 = cursor2.fetchone()
            
            if row1 != row2:
                analysis['modified_rows'].append({
                    'id': row_id,
                    'db1': row1,
                    'db2': row2
                })
    
    conn1.close()
    conn2.close()
    return analysis

# Analyze tables with significant changes
print("\n" + "="*60)
print("🔬 DETAILED TABLE ANALYSIS")
print("="*60)

row_counts = comparison_results['row_count_comparison']
changed_tables = {t: c for t, c in row_counts.items() if c['difference'] != 0}

if changed_tables:
    print(f"\nAnalyzing {len(changed_tables)} tables with changes...\n")
    for table in list(changed_tables.keys())[:5]:  # Analyze first 5 changed tables
        print(f"📊 Analyzing: {table}")
        analysis = get_table_diff_analysis(table, RELEASES['e6c2']['path'], RELEASES['e6c3']['path'])
        
        print(f"   ✓ Columns: {len(analysis['columns'])}")
        print(f"   ➕ Added rows: {len(analysis['added_rows'])}")
        print(f"   ➖ Removed rows: {len(analysis['removed_rows'])}")
        print(f"   🔄 Modified rows (sampled): {len(analysis['modified_rows'])}")
        
        if analysis['added_rows'][:5]:
            print(f"   Sample added IDs: {analysis['added_rows'][:5]}")
        if analysis['removed_rows'][:5]:
            print(f"   Sample removed IDs: {analysis['removed_rows'][:5]}")
        print()
else:
    print("✓ No tables with row count changes to analyze")

## Summary Statistics

In [ ]:
print("\n" + "="*60)
print("📈 FINAL COMPARISON SUMMARY")
print("="*60)

print(f"\n🗄️  Database Comparison: {RELEASES['e6c2']['name']} → {RELEASES['e6c3']['name']}")
print(f"\nFile Sizes:")
print(f"  E6C2: {db_info['e6c2']['size_mb']:.2f} MB")
print(f"  E6C3: {db_info['e6c3']['size_mb']:.2f} MB")
print(f"  Difference: {db_info['e6c3']['size_mb'] - db_info['e6c2']['size_mb']:+.2f} MB")

print(f"\nTable Count:")
print(f"  E6C2: {len(db_info['e6c2']['tables'])} tables")
print(f"  E6C3: {len(db_info['e6c3']['tables'])} tables")

print(f"\nTotal Rows:")
print(f"  E6C2: {db_info['e6c2']['total_rows']:,} rows")
print(f"  E6C3: {db_info['e6c3']['total_rows']:,} rows")
print(f"  Difference: {db_info['e6c3']['total_rows'] - db_info['e6c2']['total_rows']:+,} rows")

row_counts = comparison_results['row_count_comparison']
changed_count = sum(1 for c in row_counts.values() if c['difference'] != 0)
print(f"\nTables with changes: {changed_count}/{len(row_counts)}")

schema = comparison_results['schema_differences']
total_changes = len(schema['column_differences'])
print(f"Tables with schema changes: {total_changes}/{len(schema['in_both'])}")

print(f"\n✅ Comparison complete at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)